# 02_01_Profiling and Cleaning Datasets : stations and municipalities #

In [1]:
from pathlib import Path

import pandas as pd

import infrabel_punctuality as ip

In [2]:
path_raw = Path("../data/raw/")

path_intermediate = Path("../data/intermediate/")

# Table of Contents
- [1. STATION DATASET](#1-station-dataset)
- [1.1. Data profiling](#11-data-profiling)
    - [1.1.1. Overview](#111-overview)
    - [1.1.2. Profiling the `class_en` column values](#112-profiling-the-class_en-column-values)
- [1.2. Data quality and cleaning](#12-data-quality-and-cleaning)
    - [1.2.1. Handling missing values](#121-handling-missing-values)
    - [1.2.2. Converting data types](#122-converting-data-types)
    - [1.2.3. Selecting the relevant classes in the `class_en` column](#123-selecting-the-relevant-classes-in-the-class_en-column)
    - [1.2.4. Droping useless columns](#124-droping-useless-columns)
- [1.3. Exporting to Silver layer](#13-exporting-to-silver-layer)


## **1. STATION DATASET** ##

## 1.1. Data profiling ##

### 1.1.1. Overview ###

As shown below :
* `ptcarid` is unique → suitable as **business key**.

* Most descriptive columns are also unique (one row per operational point).

* `class_en` contains 12 categories and will be used **to filter relevant station types**.

* `geo_point_2d` and `geo_shape` contain **17 missing values**.

* There are no duplicate values in the dataframe.

In [3]:
stations = pd.read_parquet(path_raw / "operational_pts_railway_raw.parquet")
stations.head()

,geo_point_2d,geo_shape,ptcarid,taftapcode,symbolicname,shortnamefrench,shortnamedutch,longnamefrench,longnamedutch,commercialshortnamefrench,commercialshortnamedutch,commercialmiddlenamefrench,commercialmiddlenamedutch,commerciallongnamefrench,commerciallongnamedutch,class_en
0,b'\x01\x01\x00\x00\x00\x13\x11\xbf\xc37\xfd\x1...,b'\x01\x01\x00\x00\x00\x13\x11\xbf\xc37\xfd\x1...,443,BE00443,FKGLF,GENK-ZD-FORD,GENK-ZD-FORD,GENK-ZUID-L.O.-FORD,GENK-ZUID-L.O.-FORD,Genk-Zd-Ford,Genk-Zd-Ford,Genk-Zuid-L.O.-Ford,Genk-Zuid-L.O.-Ford,Genk-Zuid-L.O.-Ford,Genk-Zuid-L.O.-Ford,Station
1,b'\x01\x01\x00\x00\x00\r\to\xbe\xd6\xd5\x11@\x...,b'\x01\x01\x00\x00\x00\r\to\xbe\xd6\xd5\x11@\x...,275,BE00275,GCRA,CHARLEROI-AT,CHARLEROI-AT,CHARLEROI-A.T.,CHARLEROI-A.T.,Charleroi-AT,Charleroi-AT,Charleroi-A.T.,Charleroi-A.T.,Charleroi-A.T.,Charleroi-A.T.,Service installation
2,"b'\x01\x01\x00\x00\x00>\x84\x92P\xf8,\x11@\x15...","b'\x01\x01\x00\x00\x00>\x84\x92P\xf8,\x11@\x15...",1748,BE01748,VOIL1,V.OILTANK1,V.OILTANK1,VERB.OILTANKING 1,VERB.OILTANKING 1,V.Oiltank1,V.Oiltank1,Verb.Oiltanking 1,Verb.Oiltanking 1,Verb.Oiltanking 1,Verb.Oiltanking 1,Connection
3,b'\x01\x01\x00\x00\x00/J\x7f\xef4)\x11@o\xf1\x...,b'\x01\x01\x00\x00\x00/J\x7f\xef4)\x11@o\xf1\x...,1749,BE01749,VOIL2,V.OILTANK2,V.OILTANK2,VERB.OILTANKING 2,VERB.OILTANKING 2,V.Oiltank2,V.Oiltank2,Verb.Oiltanking 2,Verb.Oiltanking 2,Verb.Oiltanking 2,Verb.Oiltanking 2,Connection
4,"b'\x01\x01\x00\x00\x00\x93\xbc\xb4\xbeF""\x11@7...","b'\x01\x01\x00\x00\x00\x93\xbc\xb4\xbeF""\x11@7...",1751,BE01751,VIBR,V.IBR,V.IBR,VERB.IBR,VERB.IBR,V.IBR,V.IBR,Verb.IBR,Verb.IBR,Verb.IBR,Verb.IBR,Connection


In [4]:
stations.info()

<class 'pandas.DataFrame'>
RangeIndex: 1337 entries, 0 to 1336
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   geo_point_2d                1320 non-null   object
 1   geo_shape                   1320 non-null   object
 2   ptcarid                     1337 non-null   str   
 3   taftapcode                  1337 non-null   str   
 4   symbolicname                1337 non-null   str   
 5   shortnamefrench             1337 non-null   str   
 6   shortnamedutch              1337 non-null   str   
 7   longnamefrench              1337 non-null   str   
 8   longnamedutch               1337 non-null   str   
 9   commercialshortnamefrench   1337 non-null   str   
 10  commercialshortnamedutch    1337 non-null   str   
 11  commercialmiddlenamefrench  1337 non-null   str   
 12  commercialmiddlenamedutch   1337 non-null   str   
 13  commerciallongnamefrench    1337 non-null   str   
 14  com

In [5]:
stations.nunique()

geo_point_2d                  1320
geo_shape                     1320
ptcarid                       1337
taftapcode                    1337
symbolicname                  1337
shortnamefrench               1337
shortnamedutch                1337
longnamefrench                1337
longnamedutch                 1337
commercialshortnamefrench     1337
commercialshortnamedutch      1337
commercialmiddlenamefrench    1337
commercialmiddlenamedutch     1337
commerciallongnamefrench      1337
commerciallongnamedutch       1337
class_en                        12
dtype: int64

In [6]:
stations.isnull().sum()

geo_point_2d                  17
geo_shape                     17
ptcarid                        0
taftapcode                     0
symbolicname                   0
shortnamefrench                0
shortnamedutch                 0
longnamefrench                 0
longnamedutch                  0
commercialshortnamefrench      0
commercialshortnamedutch       0
commercialmiddlenamefrench     0
commercialmiddlenamedutch      0
commerciallongnamefrench       0
commerciallongnamedutch        0
class_en                       0
dtype: int64

In [7]:
stations.duplicated().sum()

np.int64(0)

In [8]:
stations.duplicated(subset=["ptcarid"]).sum()

np.int64(0)

In [9]:
stations["class_en"].value_counts()

class_en
Station                 421
Stop in open track      324
Junction                218
Grid                    127
Connection              101
6                        47
Service installation     35
8                        19
Net borde                15
Service stop             14
Other                    10
Movable bridge            6
Name: count, dtype: int64

### 1.1.2. Profiling the `class_en` column values ###

Classes **Station** and **Stop in open track** from the `class_en` column are retained for further analysis.

As displayed below, an inspection of the other classes reveals that they primarily correspond to technical transition points (national or linguistic boundaries, network limits, industrial dead-ends, bridges, etc.). Their relevance will be validated by cross-referencing them with the punctuality dataset.


In [10]:
classes_to_check = ["Junction", "Grid", "Connection", "6", "Service installation", "8", "Net borde", "Service stop", "Other", "Movable bridge"]

(stations[stations["class_en"].isin(classes_to_check)]
        .groupby("class_en")["commerciallongnamefrench"].agg(["count", "first"])
)


,count,first
class_en,,
6,47,Front. Bruxelles-Flandre 96L/8
8,19,Raeren-Frontière
Connection,101,Verb.Oiltanking 1
Grid,127,Monceau-Block 16
Junction,218,Y.Frederik
Movable bridge,6,Kruisschansbrug
Net borde,15,Lijn 10-Seinen KL/LL
Other,10,Genk-Zuid-LO-Inrit Zuid FORD
Service installation,35,Charleroi-A.T.


## 1.2. Data quality and cleaning ##

### 1.2.1. Handling missing values ###

* Missing values in both coordinate columns `geo_point_2d` and `geo_shape` occur **within the same 17 rows**.

* 15 Missing coordinates correspond to non-passenger operational points (connection, grid, junction, service installation).

* However, two records are labeled as `Station`:
    - Philippeville-642 (ptcarid: 2199)
    - Wondelgem-SAS636 (ptcarid: 2192)

In [11]:
stations.assign(
        geo_point_null = stations['geo_point_2d'].isnull(),
        geo_shape_null = stations['geo_shape'].isnull()
).value_counts(['geo_point_null', 'geo_shape_null']).unstack(fill_value=0)



geo_shape_null,False,True
geo_point_null,,
False,1320,0
True,0,17


In [12]:
null_in_stations = stations[["ptcarid", 
                             "class_en", 
                             "commerciallongnamefrench",
                             "commerciallongnamedutch",
                             "geo_point_2d", 
                             "geo_shape"]].loc[
    stations["geo_point_2d"].isnull() & stations["geo_shape"].isnull()
]
display(null_in_stations.style.hide())

ptcarid,class_en,commerciallongnamefrench,commerciallongnamedutch,geo_point_2d,geo_shape
2196,Connection,VERB.Polymers.II,VERB.Polymers.II,None,None
2212,Connection,Racc.Prayon,Racc.Prayon,None,None
2191,Grid,Entrée Racc.Holcim2,Entrée Racc.Holcim2,None,None
2199,Station,Philippeville-642,Philippeville-642,None,None
2213,8,Raeren-Frontière,Raeren-Frontière,None,None
2194,Connection,VERB.Taminco,VERB.Taminco,None,None
2190,Connection,Racc.Holcim2,Racc.Holcim2,None,None
2198,Connection,VERB.Disteel,VERB.Disteel,None,None
2195,Connection,VERB.Kronos,VERB.Kronos,None,None
2187,Connection,Racc.Holcim1,Racc.Holcim1,None,None


As shown in the output below, the dataset already contains valid and geolocated records for the actual passenger stations:
  - Philippeville
  - Wondelgem
  - Wondelgem-Bundel

This suggests that these records represent **technical or legacy operational points** rather than active stations used in passenger traffic.

Nevertheless, to confirm that these 17 records can be excluded from the analysis, we will compare their `ptcarid` values with those present in the punctuality dataset.

This cross-validation rests on two assumptions:
- The punctuality dataset contains operational points where train movements have been recorded over the past two years.
- If a given `ptcarid` does not appear in this dataset, it is likely not involved in regular train operations within the scope of this analysis.


In [13]:
columns_to_show = [
    "ptcarid", 
    "class_en", 
    "commerciallongnamefrench", 
    "geo_point_2d", 
    "geo_shape"
]

mask = (
    stations["commerciallongnamefrench"].str.contains("Philippeville|Wondelgem", case=False, na=False) & 
    stations["class_en"].str.contains("Station", case=False, na=False)
)

display(stations.loc[mask, columns_to_show])

,ptcarid,class_en,commerciallongnamefrench,geo_point_2d,geo_shape
8,1888,Station,Wondelgem-Bundel,b'\x01\x01\x00\x00\x00\xbe\xe17\\7\xc0\r@\x80\...,b'\x01\x01\x00\x00\x00\xbe\xe17\\7\xc0\r@\x80\...
217,2199,Station,Philippeville-642,None,None
661,2192,Station,Wondelgem-SAS636,None,None
1288,961,Station,Philippeville,b'\x01\x01\x00\x00\x00\x0f\xc6\xd3\xce\xea$\x1...,b'\x01\x01\x00\x00\x00\x0f\xc6\xd3\xce\xea$\x1...
1289,1248,Station,Wondelgem,b'\x01\x01\x00\x00\x00\x1dC\xc7\xdb\xbb\xc1\r@...,b'\x01\x01\x00\x00\x00\x1dC\xc7\xdb\xbb\xc1\r@...


**Cross-validation with the `punctuality` dataset**



To cross-validate the 17 null rows against the `punctuality` dataset, their `ptcarid` values are extracted from the `stations` dataframe and converted to integers.

In [14]:
ptcarid_rows_stations_null = stations.loc[
    (stations["geo_point_2d"].isnull()) &
    (stations["geo_shape"].isnull()),
    ["ptcarid"]
]

ptcarid_rows_stations_null = ptcarid_rows_stations_null.astype(int)

display(ptcarid_rows_stations_null)

,ptcarid
9,2196
10,2212
216,2191
217,2199
218,2213
465,2194
466,2190
467,2198
468,2195
469,2187


In [15]:
punctuality = pd.read_parquet(path_intermediate / "punctuality_raw.parquet")

In [16]:
punctuality.columns

Index(['DATDEP', 'TRAIN_NO', 'RELATION', 'TRAIN_SERV', 'PTCAR_NO', 'THOP1_COD',
       'LINE_NO_DEP', 'REAL_TIME_ARR', 'REAL_TIME_DEP', 'PLANNED_TIME_ARR',
       'PLANNED_TIME_DEP', 'DELAY_ARR', 'DELAY_DEP', 'CIRC_TYP',
       'RELATION_DIRECTION', 'PTCAR_LG_NM_NL', 'LINE_NO_ARR',
       'PLANNED_DATE_ARR', 'PLANNED_DATE_DEP', 'REAL_DATE_ARR',
       'REAL_DATE_DEP', 'OP1_COD'],
      dtype='str')

In [17]:
punctuality_ptcarid = punctuality["PTCAR_NO"].unique()

ptcarid_rows_stations_null["in_punctuality"] = (
    ptcarid_rows_stations_null["ptcarid"]
    .isin(punctuality_ptcarid)
)

with pd.option_context("display.max_rows", 20):
    display(ptcarid_rows_stations_null)

,ptcarid,in_punctuality
9,2196,False
10,2212,False
216,2191,False
217,2199,True
218,2213,False
465,2194,False
466,2190,False
467,2198,False
468,2195,False
469,2187,False


As shown above, `ptcarid` **2199** is the only one which is present in the `punctuality` dataset.  

Therefore, the others can be excluded from the `stations` dataframe.  

However, we still need to find geographical coordinates for `ptcarid` **2199** manually.

In [18]:
punctuality_ptcarid_2199 = punctuality[punctuality["PTCAR_NO"] == 2199]

In [19]:
punctuality_ptcarid_2199["RELATION_DIRECTION"].value_counts()

RELATION_DIRECTION
L C4: COUVIN -> CHARLEROI-CENTRAL    7443
L C4: CHARLEROI-CENTRAL -> COUVIN    7385
S64: COUVIN -> CHARLEROI-CENTRAL      227
S64: CHARLEROI-CENTRAL -> COUVIN      227
Name: count, dtype: int64

As shown above, `ptcarid` **2199** is an active station used in passenger traffic, on the **Couvin → Charleroi Central** and **Charleroi Central → Couvin** relations. 

After investigation, the Operational point **Philippeville-642** (`ptcarid 2199`) is identified as a technical signaling point associated with the main **Philippeville** station (`ptcarid 961`). Since these two points are functionally co-located within the same railway site, we assign the coordinates of `ptcarid` 961 to `ptcarid`2199 to ensure complete geospatial coverage in the **punctuality** analysis.

In [20]:
geo_point_2d_961 = stations.loc[stations["ptcarid"] == "961", "geo_point_2d"].iloc[0]
geo_shape_961 = stations.loc[stations["ptcarid"] == "961", "geo_shape"].iloc[0]

stations.loc[stations["ptcarid"] == "2199", "geo_point_2d"] = geo_point_2d_961
stations.loc[stations["ptcarid"] == "2199", "geo_shape"] = geo_shape_961

In [21]:
rows_to_check = ("2199", "961")

display(stations[stations["ptcarid"].isin(rows_to_check)])

,geo_point_2d,geo_shape,ptcarid,taftapcode,symbolicname,shortnamefrench,shortnamedutch,longnamefrench,longnamedutch,commercialshortnamefrench,commercialshortnamedutch,commercialmiddlenamefrench,commercialmiddlenamedutch,commerciallongnamefrench,commerciallongnamedutch,class_en
217,b'\x01\x01\x00\x00\x00\x0f\xc6\xd3\xce\xea$\x1...,b'\x01\x01\x00\x00\x00\x0f\xc6\xd3\xce\xea$\x1...,2199,BE02199,PL642,PHILIPPE642,PHILIPPE642,PHILIPPEVILLE-642,PHILIPPEVILLE-642,Philippe642,Philippe642,Philippeville-642,Philippeville-642,Philippeville-642,Philippeville-642,Station
1288,b'\x01\x01\x00\x00\x00\x0f\xc6\xd3\xce\xea$\x1...,b'\x01\x01\x00\x00\x00\x0f\xc6\xd3\xce\xea$\x1...,961,BE00961,GPL,PHILIPPEVILL,PHILIPPEVILL,PHILIPPEVILLE,PHILIPPEVILLE,Philippevill,Philippevill,Philippeville,Philippeville,Philippeville,Philippeville,Station


We can now exclude the other rows with missing coordinates from the analysis.

In [22]:
stations = stations.dropna()

As we will need these columns when we identify the relevant classes of the `class_en`column, we keep a smaller dataframe with the columns `PTCAR_NO` and `RELATION_DIRECTION` from the `punctuality` dataset.

In [23]:
punctuality_crossref = punctuality[["PTCAR_NO", "RELATION_DIRECTION"]]

We now delete the `punctuality` dataframe to avoid a `MemoryError`, but we keep the `punctuality_ptcarid` array for another cross-validation when we select the **relevant values** in the `class_en` column of the `stations` dataframe.

In [24]:
del punctuality

### 1.2.2. Converting data types ###

`PTCAR_NO` values in the `punctuality` dataframe are **integers**. For this reason, we convert the `ptcarid` column from **str** to **int**.

In [25]:
stations["ptcarid"] = stations["ptcarid"].astype(int)

In [26]:
stations.info()

<class 'pandas.DataFrame'>
Index: 1321 entries, 0 to 1336
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   geo_point_2d                1321 non-null   object
 1   geo_shape                   1321 non-null   object
 2   ptcarid                     1321 non-null   int64 
 3   taftapcode                  1321 non-null   str   
 4   symbolicname                1321 non-null   str   
 5   shortnamefrench             1321 non-null   str   
 6   shortnamedutch              1321 non-null   str   
 7   longnamefrench              1321 non-null   str   
 8   longnamedutch               1321 non-null   str   
 9   commercialshortnamefrench   1321 non-null   str   
 10  commercialshortnamedutch    1321 non-null   str   
 11  commercialmiddlenamefrench  1321 non-null   str   
 12  commercialmiddlenamedutch   1321 non-null   str   
 13  commerciallongnamefrench    1321 non-null   str   
 14  commerci

### 1.2.3. Selecting the relevant classes in the `class_en` column ###

To **select the relevant classes** in the `class_en` column and exclude the others from the analysis, we cross-validate these classes against the `punctuality` dataset, using the `punctuality_ptcarid` array created earlier.

In [27]:
stations_classes_in_punctuality = stations[stations["ptcarid"].isin(punctuality_ptcarid)].copy()

In [28]:
stations_classes_in_punctuality["class_en"].value_counts()

class_en
Station                 332
Stop in open track      324
Service installation     18
Service stop             11
Grid                      2
Name: count, dtype: int64

In [29]:
stations["class_en"].value_counts()

class_en
Station                 420
Stop in open track      324
Junction                217
Grid                    123
Connection               93
6                        47
Service installation     34
8                        18
Net borde                15
Service stop             14
Other                    10
Movable bridge            6
Name: count, dtype: int64

As the two outputs above reveal, the classes present in both dataframes are **Station**, **Stop in open track**, **Service installation**, **Service stop** and **Grid**, but the `stations` dataframe contains far more records from all these classes than the `punctuality` dataframe, except for the **Stop in open track** class. We exclude the other classes from the analysis, but retain all records from the remaining classes as they are, since having additional rows in a dimension table is not an issue in a star schema. Only orphaned records in the fact table would be problematic - a point we will investigate in the *02_04 Profiling and cleaning punctuality* notebook.

However, as there are only 2 records of 123 from the **Grid** class, we investigate these 2 records below to check if there are active stations indeed and exclude the 121 other records from this class. For this purpose, we cross-validate the `stations`and the `punctuality`dataframes.

In [30]:
classes_to_keep = ["Station", "Stop in open track", "Service installation", "Service stop", "Grid"]

stations = stations.loc[stations["class_en"].isin(classes_to_keep)].reset_index(drop=True)

In [31]:
stations["class_en"].value_counts()

class_en
Station                 420
Stop in open track      324
Grid                    123
Service installation     34
Service stop             14
Name: count, dtype: int64

In [32]:
display(stations_classes_in_punctuality[
    stations_classes_in_punctuality["class_en"] == "Grid"]
    [["ptcarid", "longnamefrench", "longnamedutch"]])

,ptcarid,longnamefrench,longnamedutch
1115,196,HAVERSIN-GRIL K,HAVERSIN-GRIL K
1186,339,MONTZEN-GRIL N,MONTZEN-GRIL N


In [33]:
ptcarid_to_check = [196, 339]

punctuality_grids = punctuality_crossref[punctuality_crossref["PTCAR_NO"].isin(ptcarid_to_check)]
punctuality_grids["RELATION_DIRECTION"].value_counts()

RELATION_DIRECTION
IC 16-1: LUXEMBOURG -> BRUSSEL-ZUID          1699
IC 16-1: BRUSSEL-ZUID -> LUXEMBOURG          1623
L 10-1: ROCHEFORT-JEMELLE -> CINEY            876
L 10-1: CINEY -> ROCHEFORT-JEMELLE            805
IC 01: OOSTENDE -> EUPEN                      660
IC 01: EUPEN -> OOSTENDE                      585
IC 12: KORTRIJK -> WELKENRAEDT                 32
IC 12: WELKENRAEDT -> KORTRIJK                 30
L L1-1: LIEGE-SAINT-LAMBERT -> AACHEN HBF      22
Name: count, dtype: int64

The two Grid points retained in the analysis are identified as active operational points regularly served by IC and L train services, with over 6,300 recorded passages in the punctuality dataset over the past two years.

We now exclude the other 121 records from the **Grid** class, then delete `punctuality_crossref`to free up memory.

In [34]:
stations = stations.loc[(stations["class_en"] != "Grid") | (stations["ptcarid"].isin([196, 339]))].reset_index(drop=True)

In [35]:
stations["class_en"].value_counts()

class_en
Station                 420
Stop in open track      324
Service installation     34
Service stop             14
Grid                      2
Name: count, dtype: int64

### 1.2.4. Droping useless columns ###

In [36]:
stations.columns


Index(['geo_point_2d', 'geo_shape', 'ptcarid', 'taftapcode', 'symbolicname',
       'shortnamefrench', 'shortnamedutch', 'longnamefrench', 'longnamedutch',
       'commercialshortnamefrench', 'commercialshortnamedutch',
       'commercialmiddlenamefrench', 'commercialmiddlenamedutch',
       'commerciallongnamefrench', 'commerciallongnamedutch', 'class_en'],
      dtype='str')

The `taftapcode` column contains the TAF/TAP TSI codes, used to ensure harmonized European exchanges between railway stakeholders. We will not need it for this analysis. Therefore, it is excluded from the `stations` dataframe.

In [37]:
stations = stations.drop(columns=["taftapcode"])

The `stations` dataframe contains 11 columns providing station names in various formats and languages.  
To determine which columns to retain for the future station dimension, we test the match count between each column 
and the `station_name` column from the SNCB passengers dataset.


In [38]:
columns_to_check = [
    "symbolicname", 
    "shortnamefrench", 
    "shortnamedutch", 
    "longnamefrench", 
    "longnamedutch",
    "commercialshortnamefrench",
    "commercialshortnamedutch",
    "commercialmiddlenamefrench",
    "commercialmiddlenamedutch",
    "commerciallongnamefrench",
    "commerciallongnamedutch"
]

stations[columns_to_check].sort_values("commerciallongnamedutch").head()

,symbolicname,shortnamefrench,shortnamedutch,longnamefrench,longnamedutch,commercialshortnamefrench,commercialshortnamedutch,commercialmiddlenamefrench,commercialmiddlenamedutch,commerciallongnamefrench,commerciallongnamedutch
458,FBC,BRAINE-LE-CT,BRAINE-LE-CT,BRAINE-LE-COMTE,BRAINE-LE-COMTE,Braine-le-Ct,'s-Gravenbr,Braine-le-Comte,'s-Gravenbrakel,Braine-le-Comte,'s-Gravenbrakel
221,FLS,AALST,AALST,AALST,AALST,Alost,Aalst,Alost,Aalst,Alost,Aalst
561,FLSK,AALST-KERREB,AALST-KERREB,AALST-KERREBROEK,AALST-KERREBROEK,Aalst-Kerreb,Aalst-Kerreb,Aalst-Kerrebroek,Aalst-Kerrebroek,Aalst-Kerrebroek,Aalst-Kerrebroek
296,FLSO,AALST-OOST,AALST-OOST,AALST-OOST,AALST-OOST,Alost-Est,Aalst-Oost,Alost-Est,Aalst-Oost,Alost-Est,Aalst-Oost
238,FLT,AALTER,AALTER,AALTER,AALTER,Aalter,Aalter,Aalter,Aalter,Aalter,Aalter


In [39]:
stations_names = stations[columns_to_check]

In [40]:
passengers_sncb = pd.read_parquet(path_intermediate / "passengers_sncb_2024.parquet")
passengers_sncb.columns

Index(['Station \nGare',
       'Gem. instappende tijdens week \nMoy. montés en semaine',
       'Gem. instappende op zaterdag\nMoy. montés le samedi',
       'Gem. instappende op zondag\nMoy. montés le dimanche'],
      dtype='str', name=0)

In [41]:
passengers_sncb = passengers_sncb.rename(columns={"Station \nGare": "station_name"})

As shown below, the passengers_sncb count 554 stations.

In [42]:
passengers_sncb["station_name"].nunique()

554

In [43]:
sncb_ref = ip.clean_column_string(passengers_sncb["station_name"])

stations_names_to_check = stations.copy()

stations_names_to_check = ip.clean_df_string(df=stations_names_to_check, columns=columns_to_check)

In [44]:
results = {}

for col in columns_to_check:
    count = stations_names_to_check[col].isin(sncb_ref).sum()
    results[col] = int(count)

display(pd.Series(results, name="match_count"))

symbolicname                    8
shortnamefrench               434
shortnamedutch                433
longnamefrench                509
longnamedutch                 508
commercialshortnamefrench     384
commercialshortnamedutch      389
commercialmiddlenamefrench    434
commercialmiddlenamedutch     441
commerciallongnamefrench      436
commerciallongnamedutch       444
Name: match_count, dtype: int64

After testing, we get the results below :

| Column                    | Matches |
|---------------------------|---------|
| symbolicname              | 3       |
| shortnamefrench           | 434     |
| shortnamedutch            | 433     |
| longnamefrench            | 509     |
| longnamedutch             | 508     |
| commercialshortnamefrench | 384     |
| commercialshortnamedutch  | 389     |
| commercialmiddlenamefrench| 434     |
| commercialmiddlenamedutch | 441     |
| commerciallongnamefrench  | 436     |
| commerciallongnamedutch   | 444     |

`longnamefrench` and `longnamedutch` yield the highest match count (509 and 508) against the 554 SNCB stations. 

**Note 1:** Some station names are identical in both languages (e.g. Brussels-area and single-language stations).  
Other stations from the passengers_sncb dataset are orphans. We will address that issue in the *02_03_profiling_and_cleaning_passengers* notebook.

**Note 2:** `clean_string()` was applied earlier solely for comparison purposes. The original casing of longnamefrench and longnamedutch is preserved in the final station dimension, as uppercase formatting is preferred for display in Power BI dashboards.

**Decision:** `longnamefrench` and `longnamedutch` are retained as the reference name columns for the future station dimension. The remaining 9 name columns are dropped.

In [45]:
stations.columns

Index(['geo_point_2d', 'geo_shape', 'ptcarid', 'symbolicname',
       'shortnamefrench', 'shortnamedutch', 'longnamefrench', 'longnamedutch',
       'commercialshortnamefrench', 'commercialshortnamedutch',
       'commercialmiddlenamefrench', 'commercialmiddlenamedutch',
       'commerciallongnamefrench', 'commerciallongnamedutch', 'class_en'],
      dtype='str')

In [46]:
columns_to_keep = [
    "geo_point_2d",
    "geo_shape",
    "ptcarid",
    "longnamefrench",
    "longnamedutch",
    "class_en"
]

stations = stations[columns_to_keep]

In [47]:
stations.head()

,geo_point_2d,geo_shape,ptcarid,longnamefrench,longnamedutch,class_en
0,b'\x01\x01\x00\x00\x00\x13\x11\xbf\xc37\xfd\x1...,b'\x01\x01\x00\x00\x00\x13\x11\xbf\xc37\xfd\x1...,443,GENK-ZUID-L.O.-FORD,GENK-ZUID-L.O.-FORD,Station
1,b'\x01\x01\x00\x00\x00\r\to\xbe\xd6\xd5\x11@\x...,b'\x01\x01\x00\x00\x00\r\to\xbe\xd6\xd5\x11@\x...,275,CHARLEROI-A.T.,CHARLEROI-A.T.,Service installation
2,b'\x01\x01\x00\x00\x00\xbe\xe17\\7\xc0\r@\x80\...,b'\x01\x01\x00\x00\x00\xbe\xe17\\7\xc0\r@\x80\...,1888,WONDELGEM-BUNDEL,WONDELGEM-BUNDEL,Station
3,b'\x01\x01\x00\x00\x00t\x87S\x00\x96\xf8\t@\xd...,b'\x01\x01\x00\x00\x00t\x87S\x00\x96\xf8\t@\xd...,2155,ZEEBRUGGE-BRAM-TANKINST.,ZEEBRUGGE-BRAM-TANKINST.,Service installation
4,b'\x01\x01\x00\x00\x00\xf0\xdc\xd0t~\xfe\t@\x7...,b'\x01\x01\x00\x00\x00\xf0\xdc\xd0t~\xfe\t@\x7...,2154,ZEEBRUGGE-BRAM-DSP880,ZEEBRUGGE-BRAM-DSP880,Station


In [48]:
stations.info()

<class 'pandas.DataFrame'>
RangeIndex: 794 entries, 0 to 793
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   geo_point_2d    794 non-null    object
 1   geo_shape       794 non-null    object
 2   ptcarid         794 non-null    int64 
 3   longnamefrench  794 non-null    str   
 4   longnamedutch   794 non-null    str   
 5   class_en        794 non-null    str   
dtypes: int64(1), object(2), str(3)
memory usage: 65.3+ KB


## 1.3. Exporting to Silver layer ##

Now, we can export the cleaned `stations` dataframe to the intermediate output directory.

In [49]:
stations.to_parquet(path_intermediate / "stations_cleaned.parquet")

In [50]:
df_to_verify = pd.read_parquet(path_intermediate / "stations_cleaned.parquet")
print(df_to_verify.shape, df_to_verify.dtypes.to_dict())

(794, 6) {'geo_point_2d': dtype('O'), 'geo_shape': dtype('O'), 'ptcarid': dtype('int64'), 'longnamefrench': <StringDtype(na_value=nan)>, 'longnamedutch': <StringDtype(na_value=nan)>, 'class_en': <StringDtype(na_value=nan)>}
